In [5]:
import psutil

# Check memory usage
def memory_usage():
    process = psutil.Process()
    mem_info = process.memory_info()
    return mem_info.rss / (1024 * 1024)  # Convert bytes to MB

print(f"Memory usage: {memory_usage()} MB")

Memory usage: 73.3984375 MB


: 

In [3]:
import pandas as pd
from fuzzywuzzy import process, fuzz
import re
from joblib import Parallel, delayed

# Ma'lumotlarni standartlashtirish funktsiyasi
def standardize_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'ʻ', '', text)
    text = re.sub(r'\bMFY\b', '', text)
    text = re.sub(r'\bQFY\b', '', text)
    text = re.sub(r'\bOFY\b', '', text)
    return text

# X ni H ga o'zgartirish funktsiyasi
def replace_x_with_h(text):
    return text.replace('x', 'h')

# Fuzzy matching yordamida name ustunlarini solishtirish
def fuzzy_name_match(name, choices, scorer=fuzz.ratio, threshold=80):
    results = process.extract(name, choices, scorer=scorer)
    for match, score in results:
        if score >= threshold:
            return match
    return None

d:\ProgrammFiles\miniconda\envs\dev\lib\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [4]:
# CSV fayllarni yuklash
mainDic = pd.read_csv('./kad_countryside.csv')
newDic = pd.read_csv('./soato_countryside.csv')

# 'district_id' va 'name' ustunlarini satrlarga o‘tkazish
mainDic['district_id'] = mainDic['district_id'].astype(str)
newDic['district_id'] = newDic['district_id'].astype(str)

# Ma'lumotlarni standartlashtirish va X harfini H ga o'zgartirish
mainDic['name_standardized'] = mainDic['countryside_name'].apply(standardize_text).apply(replace_x_with_h)
newDic['name_standardized'] = newDic['name'].apply(standardize_text).apply(replace_x_with_h)

# newDic dan mainDic dagi district_id ga teng bo'lgan qatorlarni tanlash
matched_rows = newDic[newDic['district_id'].isin(mainDic['district_id'])].copy()

# Fuzzy matching ni parallel ravishda qo'llash
choices = mainDic['name_standardized'].tolist()
matched_rows['name_match'] = Parallel(n_jobs=-1)(
    delayed(fuzzy_name_match)(row, choices) for row in matched_rows['name_standardized']
)

# Fuzzy matching natijalarini mainDic bilan birlashtirish
final_merged = pd.merge(mainDic, matched_rows, left_on=['district_id', 'name_standardized'], right_on=['district_id', 'name_match'], how='left', suffixes=('_main', '_new'))

# Natijani CSV faylga saqlash
final_merged.to_csv('./countryside_dictionary2.csv', index=False)

print("District_id bo'yicha mos kelgan va name solishtirilgan qatorlar birlashtirildi va saqlandi!")

District_id bo'yicha mos kelgan va name solishtirilgan qatorlar birlashtirildi va saqlandi!
